In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset

class MyDataset(Dataset):
  def __init__(self):
    self.X = torch.linspace(-10, 10, 300)
    self.y = self.X**2
    self.X = self.X.reshape(-1, 1)
    self.y = self.y.reshape(-1, 1)

  def __len__(self):
    return len(self.X)
  def __getitem__(self, index):
    return self.X[index], self.y[index]

dataset = MyDataset()

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)

class NeuralNetwork(nn.Module):
  def __init__(self):
    super().__init__()

    self.layer1 = nn.Linear(1, 10)
    self.activation = nn.ReLU()
    self.layer2 = nn.Linear(10,1)

  def forward(self, x):
    x = self.layer1(x)
    x = self.activation(x)
    x = self.layer2(x)

    return x

model = NeuralNetwork()

loss_function = nn.MSELoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.001
)

for epoch in range(1000):
  for X_batch, y_batch in loader:
    optimizer.zero_grad()
    prediction = model(X_batch)
    loss = loss_function(prediction, y_batch)
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
      print(
          f"Epoch: {epoch}, Loss: {loss.item():.4f}"
      )

model.eval()

test_values = torch.tensor([
    [3.0],
    [5.0],
    [7.0],
    [-3.0],
    [-5.0],
    [-7.0]
])

with torch.no_grad():
    predictions = model(test_values)

for x, prediction in zip(test_values, predictions):
    print(
        f"x = {x.item():.1f}, "
        f"Predicted = {prediction.item():.2f}, "
        f"Actual = {x.item()**2:.2f}"
    )

Epoch: 0, Loss: 2353.6455
Epoch: 0, Loss: 2134.1458
Epoch: 0, Loss: 881.5515
Epoch: 0, Loss: 1410.1064
Epoch: 0, Loss: 1534.3094
Epoch: 0, Loss: 1178.3606
Epoch: 0, Loss: 1522.8237
Epoch: 0, Loss: 875.7190
Epoch: 0, Loss: 1781.0814
Epoch: 0, Loss: 602.1323
Epoch: 100, Loss: 15.4785
Epoch: 100, Loss: 11.2721
Epoch: 100, Loss: 15.7303
Epoch: 100, Loss: 15.2379
Epoch: 100, Loss: 13.3224
Epoch: 100, Loss: 15.6738
Epoch: 100, Loss: 17.3694
Epoch: 100, Loss: 14.0081
Epoch: 100, Loss: 15.2296
Epoch: 100, Loss: 15.3125
Epoch: 200, Loss: 14.7970
Epoch: 200, Loss: 11.5808
Epoch: 200, Loss: 9.0579
Epoch: 200, Loss: 11.7541
Epoch: 200, Loss: 10.3445
Epoch: 200, Loss: 11.5905
Epoch: 200, Loss: 15.0568
Epoch: 200, Loss: 6.4058
Epoch: 200, Loss: 9.7049
Epoch: 200, Loss: 9.4651
Epoch: 300, Loss: 12.5162
Epoch: 300, Loss: 8.8414
Epoch: 300, Loss: 8.5732
Epoch: 300, Loss: 7.4389
Epoch: 300, Loss: 3.9578
Epoch: 300, Loss: 10.3090
Epoch: 300, Loss: 11.0592
Epoch: 300, Loss: 9.8764
Epoch: 300, Loss: 12.743

#🎯 Project Goal

###Build a neural network that classifies whether a student will Pass or Fail based on:



*     Hours studied
*     Attendance percentage
*     Previous exam score



###We will create a small dataset ourselves so you can focus on understanding the PyTorch workflow.

##Classification
*     0 → Fail
*     1 → Pass

In [1]:
# ============================================================
# DAY 20 — NEURAL NETWORK CLASSIFICATION SYSTEM
# ============================================================

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report


# ============================================================
# 1. DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


# ============================================================
# 2. CREATE DATASET
# ============================================================

X = torch.tensor([
    [2.0, 55.0, 40.0],
    [3.0, 60.0, 45.0],
    [1.5, 50.0, 35.0],
    [4.0, 65.0, 50.0],
    [5.0, 70.0, 55.0],
    [6.0, 75.0, 60.0],
    [7.0, 80.0, 65.0],
    [8.0, 85.0, 70.0],
    [9.0, 90.0, 80.0],
    [10.0, 95.0, 85.0],

    [2.5, 58.0, 42.0],
    [3.5, 62.0, 48.0],
    [4.5, 68.0, 52.0],
    [5.5, 72.0, 58.0],
    [6.5, 78.0, 63.0],
    [7.5, 82.0, 68.0],
    [8.5, 88.0, 75.0],
    [9.5, 92.0, 82.0],
    [1.0, 45.0, 30.0],
    [3.0, 52.0, 38.0],
    [4.0, 58.0, 45.0],
    [5.0, 64.0, 52.0],
    [6.0, 70.0, 59.0],
    [7.0, 76.0, 66.0]
], dtype=torch.float32)


# 0 = Fail
# 1 = Pass

y = torch.tensor([
    0, 0, 0, 0, 0,
    1, 1, 1, 1, 1,
    0, 0, 0, 1, 1,
    1, 1, 1, 0, 0,
    0, 0, 0, 1
], dtype=torch.long)


print("\nTotal samples:", len(X))
print("Number of features:", X.shape[1])


# ============================================================
# 3. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

# First:
# 75% → Train + Validation
# 25% → Test

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)


# Then:
# 75% of Train+Validation → Training
# 25% of Train+Validation → Validation

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.25,
    random_state=42,
    stratify=y_train_val
)


print("\nDataset Split:")
print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))


# ============================================================
# 4. FEATURE SCALING
# ============================================================

scaler = StandardScaler()

# IMPORTANT:
# Fit ONLY on training data

X_train = torch.tensor(
    scaler.fit_transform(X_train.numpy()),
    dtype=torch.float32
)

# Only transform validation and test data

X_val = torch.tensor(
    scaler.transform(X_val.numpy()),
    dtype=torch.float32
)

X_test = torch.tensor(
    scaler.transform(X_test.numpy()),
    dtype=torch.float32
)


# ============================================================
# 5. CUSTOM DATASET
# ============================================================

class MyDataset(Dataset):

    def __init__(self, X, y):

        self.X = X
        self.y = y

    def __len__(self):

        return len(self.X)

    def __getitem__(self, index):

        return self.X[index], self.y[index]


# Create datasets

train_dataset = MyDataset(X_train, y_train)
val_dataset = MyDataset(X_val, y_val)
test_dataset = MyDataset(X_test, y_test)


# ============================================================
# 6. DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False
)


# ============================================================
# 7. CREATE NEURAL NETWORK
# ============================================================

class ClassificationNetwork(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(3, 16),
            nn.ReLU(),

            nn.Linear(16, 8),
            nn.ReLU(),

            nn.Linear(8, 1)
        )

    def forward(self, x):

        return self.network(x)


model = ClassificationNetwork().to(device)

print("\nModel:")
print(model)


# ============================================================
# 8. LOSS FUNCTION
# ============================================================

loss_function = nn.BCEWithLogitsLoss()


# ============================================================
# 9. OPTIMIZER
# ============================================================

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)


# ============================================================
# 10. TRAINING + VALIDATION
# ============================================================

epochs = 100

best_val_loss = float("inf")


for epoch in range(epochs):

    # --------------------------------------------------------
    # TRAINING
    # --------------------------------------------------------

    model.train()

    train_loss = 0.0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)

        y_batch = y_batch.float().unsqueeze(1).to(device)

        # Forward pass
        predictions = model(X_batch)

        # Calculate loss
        loss = loss_function(
            predictions,
            y_batch
        )

        # Clear previous gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        train_loss += loss.item()


    train_loss /= len(train_loader)


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model.eval()

    val_loss = 0.0

    correct = 0
    total = 0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(device)

            y_batch = y_batch.float().unsqueeze(1).to(device)

            # Forward pass
            predictions = model(X_batch)

            # Validation loss
            loss = loss_function(
                predictions,
                y_batch
            )

            val_loss += loss.item()

            # Convert logits → probability
            probabilities = torch.sigmoid(predictions)

            # Probability >= 0.5 → class 1
            predicted_classes = (
                probabilities >= 0.5
            ).float()

            correct += (
                predicted_classes == y_batch
            ).sum().item()

            total += y_batch.size(0)


    val_loss /= len(val_loader)

    val_accuracy = correct / total


    # --------------------------------------------------------
    # SAVE BEST MODEL
    # --------------------------------------------------------

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            "best_model.pth"
        )

        best_model = True

    else:

        best_model = False


    # --------------------------------------------------------
    # PRINT PROGRESS
    # --------------------------------------------------------

    if (epoch + 1) % 10 == 0:

        message = (
            f"Epoch [{epoch + 1}/{epochs}] | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Accuracy: {val_accuracy:.2%}"
        )

        if best_model:
            message += " | Best Model Saved"

        print(message)


# ============================================================
# 11. LOAD BEST MODEL
# ============================================================

model.load_state_dict(
    torch.load(
        "best_model.pth",
        map_location=device,
        weights_only=True
    )
)

print("\nBest model loaded successfully.")


# ============================================================
# 12. FINAL TEST EVALUATION
# ============================================================

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)

        logits = model(X_batch)

        probabilities = torch.sigmoid(logits)

        predictions = (
            probabilities >= 0.5
        ).int()

        all_predictions.extend(
            predictions.squeeze(1).cpu().tolist()
        )

        all_labels.extend(
            y_batch.tolist()
        )


# ============================================================
# 13. ACCURACY
# ============================================================

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

print("\n==============================")
print("FINAL TEST RESULTS")
print("==============================")

print(
    f"Test Accuracy: {accuracy:.2%}"
)


# ============================================================
# 14. CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report:")

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=[
            "Fail",
            "Pass"
        ],
        zero_division=0
    )
)


# ============================================================
# 15. MAKE A NEW PREDICTION
# ============================================================

# New student:
#
# Hours studied   = 7
# Attendance      = 85%
# Previous score  = 70

new_student = torch.tensor([
    [7.0, 85.0, 70.0]
], dtype=torch.float32)


# Scale using the SAME scaler
new_student = torch.tensor(
    scaler.transform(
        new_student.numpy()
    ),
    dtype=torch.float32
).to(device)


# Prediction

model.eval()

with torch.no_grad():

    logit = model(new_student)

    probability = torch.sigmoid(logit)

    prediction = (
        probability >= 0.5
    ).int()


print("\n==============================")
print("NEW STUDENT PREDICTION")
print("==============================")

print(
    f"Pass Probability: "
    f"{probability.item():.2%}"
)

if prediction.item() == 1:

    print("Prediction: PASS")

else:

    print("Prediction: FAIL")

Using device: cpu

Total samples: 24
Number of features: 3

Dataset Split:
Training: 13
Validation: 5
Test: 6

Model:
ClassificationNetwork(
  (network): Sequential(
    (0): Linear(in_features=3, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=8, bias=True)
    (3): ReLU()
    (4): Linear(in_features=8, out_features=1, bias=True)
  )
)
Epoch [10/100] | Train Loss: 0.1353 | Val Loss: 0.1624 | Val Accuracy: 100.00% | Best Model Saved
Epoch [20/100] | Train Loss: 0.1046 | Val Loss: 0.0782 | Val Accuracy: 100.00%
Epoch [30/100] | Train Loss: 0.2716 | Val Loss: 0.0543 | Val Accuracy: 100.00%
Epoch [40/100] | Train Loss: 0.0703 | Val Loss: 0.0378 | Val Accuracy: 100.00% | Best Model Saved
Epoch [50/100] | Train Loss: 0.0476 | Val Loss: 0.0278 | Val Accuracy: 100.00%
Epoch [60/100] | Train Loss: 0.0280 | Val Loss: 0.0222 | Val Accuracy: 100.00%
Epoch [70/100] | Train Loss: 0.0149 | Val Loss: 0.0385 | Val Accuracy: 100.00%
Epoch [80/100] | Train Loss: 